In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.models import resnet50
from torchvision.models import densenet121
from PIL import Image
from pathlib import Path
import pandas as pd
import zipfile

In [ ]:
BASE = Path("/content/drive/MyDrive/ORIE 5750 - Applied Machine Learning/Final/train_val_split")

TRAIN_ZIP = BASE / "train_224.zip"
VAL_ZIP   = BASE / "val_224.zip"

TRAIN_CSV = BASE / "classification_train.csv"
VAL_CSV   = BASE / "classification_val.csv"

RESNET_DIR = Path("/content/resnet_dataset")
IMG_TRAIN = RESNET_DIR / "images/train"
IMG_VAL   = RESNET_DIR / "images/val"

IMG_TRAIN.mkdir(parents=True, exist_ok=True)
IMG_VAL.mkdir(parents=True, exist_ok=True)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

def extract_zip(zip_path, out_dir):
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(out_dir)

extract_zip(TRAIN_ZIP, IMG_TRAIN)
extract_zip(VAL_ZIP, IMG_VAL)

train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)

train_label_map = dict(zip(train_df["patientId"], train_df["label"]))
val_label_map   = dict(zip(val_df["patientId"], val_df["label"]))

In [ ]:
def build_samples(img_dir, label_map):
    samples = []
    for img_path in img_dir.rglob("*"):
        if img_path.suffix.lower() in [".jpg", ".png", ".jpeg"]:
            pid = img_path.stem
            if pid in label_map:
                samples.append((str(img_path), label_map[pid]))
    return samples

train_samples = build_samples(IMG_TRAIN, train_label_map)
val_samples   = build_samples(IMG_VAL, val_label_map)

print("Train samples:", len(train_samples))
print("Val samples:", len(val_samples))

transform_224 = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

def collate_fn(batch):
    images = []
    labels = []

    for img_path, label in batch:
        img = Image.open(img_path).convert("RGB")
        img = transform_224(img)
        images.append(img)
        labels.append(label)

    return torch.stack(images), torch.tensor(labels)

def get_resnet50_binary():
    model = resnet50(weights="IMAGENET1K_V2")
    model.fc = nn.Linear(model.fc.in_features, 1)
    return model

def get_densenet121_binary(dropout_p=0.0):
    model = densenet121(weights="IMAGENET1K_V1")

    in_features = model.classifier.in_features
    if dropout_p > 0:
        model.classifier = nn.Sequential(
            nn.Dropout(dropout_p),
            nn.Linear(in_features, 1)
        )
    else:
        model.classifier = nn.Linear(in_features, 1)

    return model

def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True).float().unsqueeze(1)

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x.size(0)
        preds = torch.sigmoid(logits) >= 0.5
        correct += (preds == y.bool()).sum().item()
        total += y.size(0)

    return total_loss / total, correct / total


def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True).float().unsqueeze(1)

            logits = model(x)
            loss = criterion(logits, y)

            total_loss += loss.item() * x.size(0)
            preds = torch.sigmoid(logits) >= 0.5
            correct += (preds == y.bool()).sum().item()
            total += y.size(0)

    return total_loss / total, correct / total

In [ ]:
criterion = nn.BCEWithLogitsLoss()

train_dl = DataLoader(
    train_samples,
    batch_size=32,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=(device.type == "cuda")
)

val_dl = DataLoader(
    val_samples,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=(device.type == "cuda")
)

modelRN50_001_32 = get_resnet50_binary().to(device)
optimizerRN50 = torch.optim.Adam(modelRN50_001_32.parameters(), lr=1e-3)

In [ ]:
EPOCHS = 10
last_val_acc = 0
print(f"Resnet50 model with LR={1e-3}, BATCH={32}:")
for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}:")
    train_loss, train_acc = train_one_epoch(modelRN50_001_32, train_dl, criterion, optimizerRN50)
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    val_loss, val_acc = evaluate(modelRN50_001_32, val_dl, criterion)
    print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")

    if val_acc > last_val_acc:
      torch.save(modelRN50_001_32, "resnet50_pneumonia_full.pth")
      print("Model saved.")
      last_val_acc = val_acc

modelRN50_0001_32 = get_resnet50_binary().to(device)
optimizerRN50 = torch.optim.Adam(modelRN50_0001_32.parameters(), lr=1e-4)

In [ ]:
EPOCHS = 10
print(f"Resnet50 model with LR={1e-4}, BATCH={32}:")
for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}:")
    train_loss, train_acc = train_one_epoch(modelRN50_0001_32, train_dl, criterion, optimizerRN50)
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    val_loss, val_acc = evaluate(modelRN50_0001_32, val_dl, criterion)
    print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")

    if val_acc > last_val_acc:
      torch.save(modelRN50_0001_32, "resnet50_pneumonia_full.pth")
      print("Model saved.")
      last_val_acc = val_acc

modelRN50_00003_32 = get_resnet50_binary().to(device)
optimizerRN50 = torch.optim.Adam(modelRN50_00003_32.parameters(), lr=3e-5)

In [ ]:
EPOCHS = 10
print(f"Resnet50 model with LR={3e-5}, BATCH={32}:")
for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}:")
    train_loss, train_acc = train_one_epoch(modelRN50_00003_32, train_dl, criterion, optimizerRN50)
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    val_loss, val_acc = evaluate(modelRN50_00003_32, val_dl, criterion)
    print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")

    if val_acc > last_val_acc:
      print("Saving model...")
      torch.save(modelRN50_00003_32, "resnet50_pneumonia_full.pth")
      print("Model saved.")
      last_val_acc = val_acc


train_dl = DataLoader(
    train_samples,
    batch_size=16,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=(device.type == "cuda")
)

val_dl = DataLoader(
    val_samples,
    batch_size=16,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=(device.type == "cuda")
)

modelRN50_001_16 = get_resnet50_binary().to(device)
optimizerRN50 = torch.optim.Adam(modelRN50_001_16.parameters(), lr=1e-3)

In [ ]:
EPOCHS = 10
print(f"Resnet50 model with LR={1e-3}, BATCH={16}:")
for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}:")
    train_loss, train_acc = train_one_epoch(modelRN50_001_16, train_dl, criterion, optimizerRN50)
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    val_loss, val_acc = evaluate(modelRN50_001_16, val_dl, criterion)
    print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")

    if val_acc > last_val_acc:
      print("Saving model...")
      torch.save(modelRN50_001_16, "resnet50_pneumonia_full.pth")
      print("Model saved.")
      last_val_acc = val_acc

modelRN50_0001_16 = get_resnet50_binary().to(device)
optimizerRN50 = torch.optim.Adam(modelRN50_0001_16.parameters(), lr=1e-4)

In [ ]:
EPOCHS = 10
print(f"Resnet50 model with LR={1e-4}, BATCH={16}:")
for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}:")
    train_loss, train_acc = train_one_epoch(modelRN50_0001_16, train_dl, criterion, optimizerRN50)
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    val_loss, val_acc = evaluate(modelRN50_0001_16, val_dl, criterion)
    print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")

    if val_acc > last_val_acc:
      print("Saving model...")
      torch.save(modelRN50_0001_16, "resnet50_pneumonia_full.pth")
      print("Model saved.")
      last_val_acc = val_acc

modelRN50_00003_16 = get_resnet50_binary().to(device)
optimizerRN50 = torch.optim.Adam(modelRN50_00003_16.parameters(), lr=3e-5)

In [ ]:
EPOCHS = 10
print(f"Resnet50 model with LR={3e-5}, BATCH={16}:")
for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}:")
    train_loss, train_acc = train_one_epoch(modelRN50_00003_16, train_dl, criterion, optimizerRN50)
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    val_loss, val_acc = evaluate(modelRN50_00003_16, val_dl, criterion)
    print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")

    if val_acc > last_val_acc:
      print("Saving model...")
      torch.save(modelRN50_00003_16, "resnet50_pneumonia_full.pth")
      print("Model saved.")
      last_val_acc = val_acc


train_dl = DataLoader(
    train_samples,
    batch_size=32,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=(device.type == "cuda")
)

val_dl = DataLoader(
    val_samples,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=(device.type == "cuda")
)

modelDN121_0001_32 = get_densenet121_binary(0.3).to(device)
optimizerDN121 = torch.optim.Adam(modelDN121_0001_32.parameters(), lr=1e-4)

In [ ]:
EPOCHS = 10
last_val_acc = 0
print(f"Densenet121 model with LR={1e-4}, BATCH={32}:")
for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    train_loss, train_acc = train_one_epoch(modelDN121_0001_32, train_dl, criterion, optimizerDN121)
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    val_loss, val_acc = evaluate(modelDN121_0001_32, val_dl, criterion)
    print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")

    if val_acc > last_val_acc:
      print("Saving model...")
      torch.save(modelDN121_0001_32, "densenet121_pneumonia_full.pth")
      print("Model saved.")
      last_val_acc = val_acc

modelDN121_00003_32 = get_densenet121_binary(0.3).to(device)
optimizerDN121 = torch.optim.Adam(modelDN121_00003_32.parameters(), lr=3e-5)

In [ ]:
EPOCHS = 10
print(f"Densenet121 model with LR={3e-5}, BATCH={32}:")
for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    train_loss, train_acc = train_one_epoch(modelDN121_00003_32, train_dl, criterion, optimizerDN121)
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    val_loss, val_acc = evaluate(modelDN121_00003_32, val_dl, criterion)
    print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")

    if val_acc > last_val_acc:
      print("Saving model...")
      torch.save(modelDN121_00003_32, "densenet121_pneumonia_full.pth")
      print("Model saved.")
      last_val_acc = val_acc


train_dl = DataLoader(
    train_samples,
    batch_size=16,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=(device.type == "cuda")
)

val_dl = DataLoader(
    val_samples,
    batch_size=16,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=(device.type == "cuda")
)

modelDN121_0001_16 = get_densenet121_binary(0.3).to(device)
optimizerDN121 = torch.optim.Adam(modelDN121_0001_16.parameters(), lr=1e-4)

In [ ]:
EPOCHS = 10
print(f"Densenet121 model with LR={1e-4}, BATCH={16}:")
for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    train_loss, train_acc = train_one_epoch(modelDN121_0001_16, train_dl, criterion, optimizerDN121)
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    val_loss, val_acc = evaluate(modelDN121_0001_16, val_dl, criterion)
    print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")

    if val_acc > last_val_acc:
      print("Saving model...")
      torch.save(modelDN121_0001_16, "densenet121_pneumonia_full.pth")
      print("Model saved.")
      last_val_acc = val_acc

modelDN121_00003_16 = get_densenet121_binary(0.3).to(device)
optimizerDN121 = torch.optim.Adam(modelDN121_00003_16.parameters(), lr=3e-5)

In [ ]:
EPOCHS = 10
print(f"Densenet121 model with LR={3e-5}, BATCH={16}:")
for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    train_loss, train_acc = train_one_epoch(modelDN121_00003_16, train_dl, criterion, optimizerDN121)
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    val_loss, val_acc = evaluate(modelDN121_00003_16, val_dl, criterion)
    print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")

    if val_acc > last_val_acc:
      print("Saving model...")
      torch.save(modelDN121_00003_16, "densenet121_pneumonia_full.pth")
      print("Model saved.")
      last_val_acc = val_acc